# EP1 - Limpieza inicial (Colab / Jupyter)

Este notebook ejecuta el proceso de limpieza inicial y genera: `data/processed/movies_processed.csv`, `data/processed/tv_processed.csv` y `documentation/markdown/EP1_data_cleaning.md`.

Instrucciones rápidas:
- En Colab: sube los dos CSV (`netflix_movies_detailed_up_to_2025.csv` y `netflix_tv_shows_detailed_up_to_2025.csv`) usando el botón
 o el siguiente comando de subida.
- En Jupyter local: asegúrate de ejecutar el notebook desde la raíz del repo (donde está la carpeta `data/`).

In [6]:
import os
import sys

movies_path = None
tv_path = None

if "google.colab" in sys.modules:
    colab_movie_path = "/content/data/raw/netflix_movies_detailed_up_to_2025.csv"
    colab_tv_path = "/content/data/raw/netflix_tv_shows_detailed_up_to_2025.csv"
    if os.path.exists(colab_movie_path) and os.path.exists(colab_tv_path):
        movies_path = colab_movie_path
        tv_path = colab_tv_path
        print(f"Usando archivos en Colab: {movies_path} , {tv_path}")
    else:
        print("No se detectaron los CSV en /content/data/raw/.")
else:
    notebook_dir = os.getcwd()
    candidate_roots = [
        notebook_dir,
        os.path.abspath(os.path.join(notebook_dir, "..")),
    ]
    for candidate_root in candidate_roots:
        candidate_movies = os.path.join(
            candidate_root, "data", "netflix_movies_detailed_up_to_2025.csv"
        )
        candidate_tv = os.path.join(
            candidate_root, "data", "netflix_tv_shows_detailed_up_to_2025.csv"
        )
        if os.path.exists(candidate_movies) and os.path.exists(candidate_tv):
            movies_path = candidate_movies
            tv_path = candidate_tv
            break
    if movies_path is None:
        print("Advertencia: no se encontraron los CSV en las rutas locales esperadas.")

if movies_path is None or tv_path is None:
    raise FileNotFoundError(
        "No se localizaron ambos CSV de entrada. Revisa la carpeta data/."
    )

print("Ruta final de películas:", movies_path)
print("Ruta final de series de TV:", tv_path)

Ruta final de películas: c:\Users\shein\OneDrive\Documentos\GitHub\visualizacion-de-datos-StreamView-Analytics\data\netflix_movies_detailed_up_to_2025.csv
Ruta final de series de TV: c:\Users\shein\OneDrive\Documentos\GitHub\visualizacion-de-datos-StreamView-Analytics\data\netflix_tv_shows_detailed_up_to_2025.csv


### Función de Limpieza y Procesamiento de Datos

Para hacer el proceso de limpieza más claro, modular y reutilizable, he encapsulado toda la lógica de limpieza en una función llamada `clean_and_process_dataframe`. Esta función realiza los siguientes pasos:

- **Carga de datos**: Lee el archivo CSV especificado.
- **Detección y Eliminación de Duplicados**: Identifica y elimina filas duplicadas, tanto filas completas como duplicados basados en `show_id` si la columna existe.
- **Limpieza de Columnas de Texto**: Elimina espacios en blanco y convierte cadenas 'nan' a `pd.NA` en columnas de tipo `object`.
- **Conversión de Fechas**: Parsea la columna `date_added` a formato de fecha y maneja errores de conversión.
- **Capitalización de País**: Convierte la columna `country` a formato de título (primera letra en mayúscula).
- **Manejo de Valores Nulos**: Reemplaza cadenas vacías, 'None' y 'nan' con `pd.NA`.
- **Cálculo de Nulos**: Cuenta los valores nulos por columna para un resumen detallado.
- **Guardar Archivo Procesado**: Guarda el DataFrame limpio en un nuevo archivo CSV en el directorio `data/processed/`.

La función también devuelve un diccionario con un resumen completo de la limpieza realizada, incluyendo el número de filas originales, duplicados encontrados, filas después de la deduplicación y el conteo de valores nulos.

In [8]:
import pandas as pd
import os
from datetime import datetime

def clean_and_process_dataframe(file_path, key, processed_dir):
    """
    Loads, cleans, and processes a single Netflix dataset (movies or TV shows).

    Args:
        file_path (str): The path to the CSV file.
        key (str): A key identifying the dataset (e.g., 'movies', 'tv').
        processed_dir (str): The directory to save the processed file.

    Returns:
        tuple: A tuple containing:
            - pd.DataFrame: The cleaned DataFrame.
            - dict: A dictionary with summary information about the cleaning process.
    """
    info = {'path': file_path}

    if not file_path or not os.path.exists(file_path):
        info['error'] = 'file not found or path is None'
        # Return empty DataFrame and info for consistency
        return pd.DataFrame(), info

    df = pd.read_csv(file_path, low_memory=False)
    info['original_rows'] = int(len(df))

    # Identify and remove duplicates
    full_dup = int(df.duplicated().sum())
    info['full_row_duplicates'] = full_dup
    if 'show_id' in df.columns:
        id_dup = int(df.duplicated(subset=['show_id']).sum())
        info['show_id_duplicates'] = id_dup
    else:
        info['show_id_duplicates'] = None
    df = df.drop_duplicates()
    info['rows_after_dedup'] = int(len(df))

    # Clean object columns: strip whitespace and convert 'nan' strings to pd.NA
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    for c in obj_cols:
        # Ensure the column exists and is not entirely NA after initial processing
        if c in df.columns and not df[c].isnull().all():
            df[c] = df[c].astype(str).str.strip()
            df.loc[df[c] == 'nan', c] = pd.NA

    # Convert 'date_added' to datetime, coercing errors
    if 'date_added' in df.columns:
        info['date_added_parsed_nulls_before'] = int(df['date_added'].isnull().sum())
        df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
        info['date_added_parsed_nulls_after'] = int(df['date_added'].isnull().sum())

    # Capitalize 'country'
    if 'country' in df.columns:
        df['country'] = df['country'].where(df['country'].isna(), df['country'].str.title())

    # Replace specific string representations of nulls with pd.NA
    df = df.replace({'': pd.NA, 'None': pd.NA, 'nan': pd.NA})

    # Calculate null counts
    null_counts = df.isnull().sum().sort_values(ascending=False)
    info['null_counts_top10'] = null_counts.head(10).to_dict()
    info['total_nulls'] = int(null_counts.sum())

    # Save processed file
    out_path = os.path.join(processed_dir, f'{key}_processed.csv')
    df.to_csv(out_path, index=False)
    info['processed_path'] = out_path
    info['processed_rows'] = int(len(df))

    return df, info

In [9]:
import os
import pandas as pd
from datetime import datetime, timezone

# Resolver la raíz aunque Jupyter use notebooks/ como directorio de trabajo.
working_dir = os.getcwd()
ROOT = (
    os.path.abspath(os.path.join(working_dir, ".."))
    if os.path.basename(working_dir).lower() == "notebooks"
    else working_dir
)
processed_dir = os.path.join(ROOT, "data", "processed")
docs_dir = os.path.join(ROOT, "documentation", "markdown")

os.makedirs(processed_dir, exist_ok=True)
os.makedirs(docs_dir, exist_ok=True)

files_to_process = [("movies", movies_path), ("tv", tv_path)]
summary = {"run_date": datetime.now(timezone.utc).isoformat() + "Z", "files": {}}

for key, path in files_to_process:
    df_cleaned, info = clean_and_process_dataframe(path, key, processed_dir)
    summary["files"][key] = info

print("Procesamiento completo. Archivos guardados en:", processed_dir)
print("Resumen de la limpieza:", summary)

C:\Users\shein\AppData\Local\Temp\ipykernel_4764\3337562246.py:41: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include=['object']).columns.tolist()
C:\Users\shein\AppData\Local\Temp\ipykernel_4764\3337562246.py:41: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/u

Procesamiento completo. Archivos guardados en: c:\Users\shein\OneDrive\Documentos\GitHub\visualizacion-de-datos-StreamView-Analytics\data\processed
Resumen de la limpieza: {'run_date': '2026-08-28T19:42:09.027233+00:00Z', 'files': {'movies': {'path': 'c:\\Users\\shein\\OneDrive\\Documentos\\GitHub\\visualizacion-de-datos-StreamView-Analytics\\data\\netflix_movies_detailed_up_to_2025.csv', 'original_rows': 16000, 'full_row_duplicates': 0, 'show_id_duplicates': 0, 'rows_after_dedup': 16000, 'date_added_parsed_nulls_before': 0, 'date_added_parsed_nulls_after': 0, 'null_counts_top10': {'duration': 16000, 'country': 466, 'cast': 204, 'director': 132, 'description': 132, 'genres': 107, 'type': 0, 'show_id': 0, 'title': 0, 'date_added': 0}, 'total_nulls': 17041, 'processed_path': 'c:\\Users\\shein\\OneDrive\\Documentos\\GitHub\\visualizacion-de-datos-StreamView-Analytics\\data\\processed\\movies_processed.csv', 'processed_rows': 16000}, 'tv': {'path': 'c:\\Users\\shein\\OneDrive\\Documentos\\

In [10]:
# Escribir reporte MD con resumen de la limpieza
import os
from datetime import datetime, timezone # Added timezone import

md_path = os.path.join(docs_dir, 'EP1_data_cleaning.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('---\n')
    f.write("title: \"EP1 — Data cleaning inicial\"\n")
    f.write("author: \"Equipo StreamView (notebook)\"\n")
    f.write(f"date: {datetime.now(timezone.utc).date()}\n") # Corrected to use timezone.utc
    f.write('source_files:\n')
    # The 'files' variable used here refers to a tuple list, not the dict in summary
    # Re-using files_to_process from cell 2b1f1917 to list original source files
    for _, p in files_to_process:
        f.write(f"  - {p}\n")
    f.write('---\n\n')
    f.write('# Resumen de la limpieza inicial\n\n')
    f.write(f'Fecha de ejecución (UTC): {summary["run_date"]}\n\n')
    for key, info in summary['files'].items():
        f.write(f'## Archivo: {key}\n\n')
        if 'error' in info:
            f.write(f'- Error: {info["error"]}\n\n')
            continue
        f.write(f'- Ruta original: {info["path"]}\n')
        f.write(f'- Filas originales: {info.get("original_rows")}\n')
        f.write(f'- Filas después de eliminar duplicados: {info.get("rows_after_dedup")}\n')
        f.write(f'- Duplicados (filas completas): {info.get("full_row_duplicates")}\n')
        if info.get('show_id_duplicates') is not None:
            f.write(f'- Duplicados por `show_id`: {info.get("show_id_duplicates")}\n')
        if 'date_added_parsed_nulls_before' in info:
            f.write(f'- `date_added` nulos antes: {info.get("date_added_parsed_nulls_before")}\n')
            f.write(f'- `date_added` nulos después de parseo: {info.get("date_added_parsed_nulls_after")}\n')
        f.write(f'- Filas procesadas guardadas en: {info.get("processed_path")}\n')
        f.write(f'- Total de valores nulos (suma por columnas): {info.get("total_nulls")}\n')
        f.write('\n')
        f.write('### Top 10 columnas por valores faltantes\n\n')
        f.write('| Columna | Nulos |\n')
        f.write('|---|---:|\n')
        for col, n in info.get('null_counts_top10', {}).items():
            f.write(f'| {col} | {n} |\n')
        f.write('\n')
print(f'Reporte escrito en: {md_path}')

Reporte escrito en: c:\Users\shein\OneDrive\Documentos\GitHub\visualizacion-de-datos-StreamView-Analytics\documentation\markdown\EP1_data_cleaning.md
